# 06. 단계적 공개와 설계 비교

단계적 공개는 문서 ID와 제목만 먼저 보고, 질문에 필요한 문서를 고른 뒤, 그 문서의 자세한 본문만 모델에 전달하는 방식입니다.

## 실행 환경 준비

실습 파일은 `labs/part2/ch08-context-engineering/notebooks/06-단계적-공개와-설계-비교.ipynb`입니다. 새 커널에서 시작해 첫 코드 셀로 패키지 버전을 맞춘 뒤, 설명과 코드를 위에서부터 순서대로 실행합니다. `.env`는 현재 작업 폴더의 상위 경로에서도 찾습니다.

In [1]:
# %pip은 현재 노트북 커널에 패키지를 설치하는 Jupyter 전용 명령입니다.
# -q는 설치 로그를 줄이고, == 뒤 숫자는 수업에서 검증한 버전을 고정합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


## 1. API 연결 준비

In [2]:
# find_dotenv는 현재 폴더부터 상위 폴더를 검색해 .env 경로를 찾습니다.
from dotenv import find_dotenv, load_dotenv
import os

# usecwd=True는 노트북을 실행한 폴더를 검색 시작점으로 지정합니다.
# 키 값은 출력하지 않고 존재 여부만 확인합니다.
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
print(".env와 API 키 확인 완료")

.env와 API 키 확인 완료


### 모델 객체를 만듭니다

`.env` 확인이 끝난 뒤 모델 이름과 출력 제한을 지정합니다. 이 셀은 아직 API에 질문을 보내지 않습니다.

In [3]:
# init_chat_model은 provider:model 형식으로 채팅 모델 객체를 만듭니다.
# model은 이후 셀에서 invoke()를 호출할 때 사용합니다.
from langchain.chat_models import init_chat_model

# openai:는 공급자 이름이며, reasoning_effort="none"은 별도 추론 토큰을 사용하지 않는 설정입니다.
# max_tokens=300은 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model(
    "openai:gpt-5.4-mini",
    reasoning_effort="none",
    max_tokens=300,
)
print("LangChain 모델 준비 완료")

LangChain 모델 준비 완료


## 2. 목록과 원문을 분리합니다

목록에는 여러 문서의 ID와 제목만 넣어 선택을 돕습니다. 답에 필요한 근거는 선택한 문서의 원문에서 가져옵니다.

In [4]:
# documents는 ID·제목·원문을 가진 문서 목록입니다.
documents = [
    {"id": "E01-p2", "title": "이용 시간", "text": "월요일 휴관"},
    {"id": "E01-p3", "title": "대출 규정",
     "text": "7권을 14일 대출. 예약이 없으면 1회 7일 연장."},
    {"id": "E01-p5", "title": "주차 안내", "text": "90분 무료"},
]
print("전체 문서 수:", len(documents))

전체 문서 수: 3


### 먼저 제목 목록만 공개합니다

원문 전체를 읽기 전에 ID와 제목만 복사합니다. 반복문에서 만든 `item`을 `catalog`에 하나씩 추가합니다.

In [5]:
# catalog에는 선택에 필요한 두 필드만 남깁니다.
catalog = []
for document in documents:
    item = {"id": document["id"], "title": document["title"]}
    catalog.append(item)

print(catalog)

[{'id': 'E01-p2', 'title': '이용 시간'}, {'id': 'E01-p3', 'title': '대출 규정'}, {'id': 'E01-p5', 'title': '주차 안내'}]


## 3. 필요한 ID의 원문만 공개합니다

이 예제에서는 학생이 `E01-p3`을 직접 선택합니다.

In [6]:
# selected_id는 질문에 맞춰 고른 문서 ID입니다.
# 반복문은 ID가 같은 문서를 찾으면 selected에 저장하고 중단합니다.
selected_id = "E01-p3"
selected = None
for document in documents:
    if document["id"] == selected_id:
        selected = document
        break

if selected is None:
    raise ValueError("선택한 문서를 찾지 못했습니다.")

print("선택한 제목:", selected["title"])

선택한 제목: 대출 규정


### 세 가지 공개 방식의 길이를 비교합니다

전체 원문, 선택한 원문, 사람이 작성한 압축문을 같은 문자 수 기준으로 비교합니다.

In [7]:
# full_lines를 만든 뒤 join으로 출처가 붙은 전체 원문을 연결합니다.
full_lines = []
for document in documents:
    line = f"[{document['id']}] {document['text']}"
    full_lines.append(line)

full = "\n".join(full_lines)
compressed = "7권, 14일, 예약 없으면 1회 7일 연장 [E01-p3]"
print("full 문자 수:", len(full))
print("selected 문자 수:", len(selected["text"]))
print("compressed 문자 수:", len(compressed))

full 문자 수: 70
selected 문자 수: 29
compressed 문자 수: 33


## 4. 세 설계를 같은 조건으로 실행합니다

전체 원문, 선택 원문, 압축문을 이름별로 준비합니다. 선택 원문에도 같은 출처 ID를 붙여 세 입력의 표기 조건을 맞춥니다.

In [8]:
# design_contexts의 값만 다르고 질문과 지시는 다음 셀에서 고정합니다.
selected_context = f"[{selected['id']}] {selected['text']}"
design_contexts = {
    "full": full,
    "selected": selected_context,
    "compressed": compressed,
}
for name, context in design_contexts.items():
    print(name, "문자 수:", len(context))

full 문자 수: 70
selected 문자 수: 38
compressed 문자 수: 33


### 같은 질문과 지시로 세 번 호출합니다

압축문은 앞에서 사람이 작성한 값이므로 별도의 압축 모델 호출 비용은 없습니다. 여기서 측정하는 사용량은 세 답변 호출의 입력 토큰입니다.

In [9]:
# 모든 설계에 같은 instruction과 question을 사용합니다.
instruction = "근거의 조건·예외·출처를 빠뜨리지 마세요."
question = "대출 권수, 기간, 연장 조건과 출처를 알려 주세요."
design_results = {}
for name, context in design_contexts.items():
    messages = [("system", instruction), ("user", f"근거: {context}\n질문: {question}")]
    response = model.invoke(messages)
    usage = response.usage_metadata
    if usage is None:
        raise RuntimeError("모델 사용량 메타데이터가 없습니다.")
    design_results[name] = (response, usage)

### 실제 출력과 입력 토큰을 관찰합니다

설계 이름 아래에 입력 토큰과 답변 원문을 함께 출력합니다. 세 응답의 표현은 실행마다 달라질 수 있습니다.

In [10]:
# 튜플의 두 값을 response와 usage로 나누어 받습니다.
for name, result in design_results.items():
    response, usage = result
    print(f"\n[{name}] input_tokens={usage['input_tokens']}")
    print(response.text)


[full] input_tokens=99
대출 관련 정보는 다음과 같습니다.

- **대출 권수**: 제시된 근거에는 **대출 가능 권수**가 직접 나오지 않습니다.  
  - 출처: 제공된 근거 **[E01-p3]**에는 “7권을 14일 대출”이라고만 되어 있어, 이는 **한 번에 7권 대출 가능**하다는 뜻으로 해석할 수는 있지만, **정확히 ‘최대 대출 권수’로 명시된 표현은 아닙니다**.

- **대출 기간**: **14일**
  - 출처: **[E01-p3] 7권을 14일 대출**

- **연장 조건**: **예약이 없을 때 1회 7일 연장 가능**
  - 출처: **[E01-p3] 예약이 없으면 1회 7일 연장**

참고로,
- **[E01-p2] 월요일 휴관**
- **[E01-p5] 90분 무료**

는 대출 정보가 아니라 다른 이용 안내입니다.

원하시면 제가 이 내용을 **“대출 권수 / 기간 / 연장 조건 / 출처” 표 형식**으로 다시 정리해 드릴게요.

[selected] input_tokens=79
근거 [E01-p3]에 따르면:

- **대출 권수**: **7권**
- **기본 대출 기간**: **14일**
- **연장 조건**: **예약이 없을 경우 1회에 한해 7일 연장 가능**
- **출처**: **[E01-p3]**

즉, **7권을 14일 동안 대출할 수 있고, 해당 도서에 예약이 없으면 1회 7일 연장**됩니다.

[compressed] input_tokens=75
대출 조건은 다음과 같습니다.

- **대출 권수:** **7권**
- **대출 기간:** **14일**
- **연장 조건:** **예약이 없을 경우 1회에 한해 7일 연장 가능**
- **출처:** **E01-p3**

정리하면, **7권을 14일간 대출할 수 있고, 예약이 없는 경우에만 1회 7일 연장**됩니다.


### 필요한 정보가 보존됐는지 비교합니다

토큰 수가 작아도 대출 조건, 연장 예외, 출처가 빠지면 충분한 답이 아닙니다. 자동 검사는 문자열 존재 여부만 보여 주므로 답변 원문도 함께 읽습니다.

In [11]:
# 세 기준이 답변에 나타났는지 설계별로 확인합니다.
for name, result in design_results.items():
    response, usage = result
    answer = response.text
    checks = {
        "대출 조건": "7" in answer and "14" in answer,
        "연장 예외": "연장" in answer,
        "출처": "E01" in answer,
    }
    print(name, checks)

full {'대출 조건': True, '연장 예외': True, '출처': True}
selected {'대출 조건': True, '연장 예외': True, '출처': True}
compressed {'대출 조건': True, '연장 예외': True, '출처': True}


## 5. 목록 공개 수 한 값만 바꿉니다

전체 문서와 선택 결과는 그대로 두고 목록에서 보여 줄 항목 수만 `1`에서 `2`로 바꿉니다.

In [12]:
# 슬라이스 [:N]은 앞에서 N개 항목을 반환합니다.
base_limit = 1
changed_limit = 2
print("기본 목록:", catalog[:base_limit])
print("변경 목록:", catalog[:changed_limit])

기본 목록: [{'id': 'E01-p2', 'title': '이용 시간'}]
변경 목록: [{'id': 'E01-p2', 'title': '이용 시간'}, {'id': 'E01-p3', 'title': '대출 규정'}]


## 정리

목록, 선택, 원문 읽기를 순서대로 수행합니다. 토큰 감소와 필요한 조건의 보존 여부를 함께 비교합니다.

## 실패 재현: 질문과 다른 문서를 선택합니다

대출 질문에 주차 문서를 선택하면 입력은 짧아져도 필요한 대출 근거가 사라집니다.

In [13]:
# selected_id는 실제 선택값, required_id는 질문에 필요한 문서 ID입니다.
# 두 값이 다르면 잘못된 문서 선택이므로 ValueError를 발생시킵니다.
def require_document(selected_id: str, required_id: str) -> None:
    if selected_id != required_id:
        raise ValueError(f"필요한 문서 {required_id}가 선택되지 않았습니다.")

### 주차 문서 ID로 실패를 관찰합니다

필요한 ID `E01-p3`은 그대로 두고 선택값만 `E01-p5`로 바꿉니다. 오류에서 필요한 ID를 확인합니다.

In [14]:
# 첫 인자는 실제 선택값, 둘째 인자는 필요한 값입니다.
try:
    require_document("E01-p5", "E01-p3")
except ValueError as error:
    print("관찰한 실패:", error)
else:
    raise AssertionError("문서 선택 오류가 발생해야 합니다.")

관찰한 실패: 필요한 문서 E01-p3가 선택되지 않았습니다.


## 수정: 대출 규정 문서를 선택합니다

질문에 필요한 `E01-p3`을 선택하고 해당 원문을 확인합니다.

In [15]:
# fixed_id를 필요한 대출 규정 ID로 고친 뒤 선택 검사를 다시 실행합니다.
# 반복문으로 실제 문서를 찾아 제목도 대출 규정인지 확인합니다.
fixed_id = "E01-p3"
require_document(fixed_id, "E01-p3")
fixed_document = None
for document in documents:
    if document["id"] == fixed_id:
        fixed_document = document
        break

if fixed_document is None:
    raise ValueError("수정한 문서를 찾지 못했습니다.")

assert fixed_document["title"] == "대출 규정"
print("수정 후 선택:", fixed_document["id"], fixed_document["title"])

수정 후 선택: E01-p3 대출 규정
